**Monte Carlo estimation of $\pi$ using a pseudo-random number generator.**
A unit circle (radius 1) inscribed in a $2\times 2$ square has areas:

$$A_{\text{circle}} = \pi r^2 = \pi
\qquad
A_{\text{square}} = (2r)^2 = 4$$

If we scatter random points uniformly over the square, the fraction that
land inside the circle equals the ratio of areas:

$$\frac{\text{dots inside}}{\text{dots total}} \approx \frac{\pi}{4}
\quad\Rightarrow\quad
\pi \approx 4 \times \frac{\text{dots inside}}{\text{dots total}}$$

A point $(x, y)$ is inside the unit circle if and only if $x^2 + y^2 \leq 1$.
The estimate improves as $1/\sqrt{N}$ (the standard Monte Carlo convergence rate),
so quadrupling the number of samples halves the expected error.

---
**Defining the sampling region.**
`matplotlib.patches.Rectangle` is a convenient way to define the bounding box:
its `get_bbox()` method returns an object with `.x0`, `.y0`, `.width`, and `.height`
that will be used throughout to keep the sample geometry in one place.

In [ ]:
"""mc_circle_prng.ipynb"""

# Cell 01 - Define the sample area and total number of random points

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle, Rectangle

# Bounding box: the [-1, 1] x [-1, 1] square enclosing the unit circle
bbox = Rectangle((-1, -1), 2, 2).get_bbox()
print(bbox)

total_dots = 320 * 320  # 102,400 sample points
print(f"{total_dots = :,}")

---
**Generating random sample points.**
`np.random.default_rng(seed=2020)` creates a **PCG64** pseudo-random number
generator (PRNG) seeded for reproducibility.
`rng.random()` returns values in the half-open interval $[0, 1)$.
Multiplying by the box width/height and adding the anchor coordinate
scales each value into $[-1, 1)$.

The interval is half-open: $-1$ is attainable but $+1$ is not.
A *closed* $[-1, 1]$ cannot be produced by rescaling `rng.random()` at all -
any affine map of a half-open interval is still half-open,
so you may choose which endpoint is included, but never both.
(A closed interval would require drawing 53-bit integers with
`rng.integers(..., endpoint=True)` and dividing by $2^{53}$.)

The distinction has no effect on the estimate.
A sample lands *on* the circle only when $x^2 + y^2 = 1$ holds exactly in
floating point, which happens with probability $\approx 0$;
the endpoint $x = 1$ would matter only if $y$ were exactly $0$ as well.

In [ ]:
# Cell 02 - Generate random (x, y) points uniformly over the bounding box

rng = np.random.default_rng(seed=2020)

x = rng.random(total_dots) * bbox.width + bbox.x0  # x in [-1, 1)
y = rng.random(total_dots) * bbox.height + bbox.y0  # y in [-1, 1)

pd.DataFrame({"x": x[:5], "y": y[:5]})

---
**Distance from the origin.**
For each point $(x_i, y_i)$ the Euclidean distance from the origin is
$d_i = \sqrt{x_i^2 + y_i^2}$.
NumPy's vectorized arithmetic computes all 102,400 distances in one expression
with no Python loop.

In [ ]:
# Cell 03 - Compute distance from origin for every point

d = np.sqrt(x**2 + y**2)

pd.DataFrame({"x": x[:5], "y": y[:5], "d": d[:5]})

---
**Classifying points as inside or outside the circle.**
Boolean indexing with `d <= 1.0` filters the coordinate arrays in a single
vectorized step - no loop needed.
Points exactly on the circumference ($d = 1$) are counted as inside.

In [ ]:
# Cell 04 - Partition points into inside (d <= 1) and outside (d > 1) the circle

x_in = x[d <= 1.0]
y_in = y[d <= 1.0]
x_out = x[d > 1.0]
y_out = y[d > 1.0]

pd.DataFrame(
    {"x_in": x_in[:5], "y_in": y_in[:5], "x_out": x_out[:5], "y_out": y_out[:5]}
)

---
**Scatter plot of the Monte Carlo sample.**
Red dots landed inside the unit circle; blue dots landed outside.
The black circle marks the exact boundary $x^2 + y^2 = 1$.

In [ ]:
# Cell 05 - Scatter plot: red = inside, blue = outside, black circle = exact boundary

plt.figure(figsize=(7, 7))
plt.scatter(x_in, y_in, color="red", marker=".", s=0.5)
plt.scatter(x_out, y_out, color="blue", marker=".", s=0.5)
plt.gca().add_patch(Circle((0, 0), 1, fill=False, color="black", linewidth=1.5))
plt.gca().set_aspect("equal")
plt.title(f"Monte Carlo Circle ({total_dots:,} samples)")
plt.xlabel("x")
plt.ylabel("y")
plt.tight_layout()
plt.show()

---
**Estimating $\pi$ and measuring the error.**
The estimated area of the circle is the fraction of points inside
multiplied by the total area of the square:

$$\hat{\pi} = A_{\text{square}} \times \frac{\text{dots inside}}{\text{dots total}}
= 4 \times \frac{\text{dots inside}}{\text{dots total}}$$

The reported error is the **absolute percent relative error**:

$$\varepsilon = \left|\frac{\hat{\pi} - \pi}{\pi}\right| \times 100\%$$

It is *relative* because the difference is divided by the true value,
and *absolute* because the sign is discarded.
Note this is not the absolute error $|\hat{\pi} - \pi|$,
which for this run is about $0.0029$ rather than $0.09\%$.

In [ ]:
# Cell 06 - Compute the pi estimate and absolute percent relative error

act = np.pi
est = (bbox.width * bbox.height) * np.count_nonzero(d <= 1.0) / total_dots
err = np.abs((est - act) / act)

print(f"dots total        : {total_dots:,}")
print(f"dots inside       : {np.count_nonzero(d <= 1.0):,}")
print(f"actual pi         : {act:.6f}")
print(f"estimated         : {est:.6f}")
print(f"abs pct rel error : {err:.5%}")